## Investigation of the reliability of T1w images

In [7]:
import re
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from bids import BIDSLayout
from nipype.interfaces.ants import ApplyTransforms

data_path = Path("/data/derivatives/hcph-smriprep-reliability")
layout = BIDSLayout(data_path / "smriprep", validate=False)

## Preprocessing

Smriprep does not directly output the individual T1w image that have been corrected for intensity non-uniformity (INU), it only outputs the preprocessed T1w combining all T1w from the same subject, which is the reference T1w image for alignement. To get access to the bias-field corrected individual T1w, we copied them from smriprep work directory into `inu_corrected_individual_T1w`. Specifially, they were stored in `$WORKDIR/smriprep_wf/single_subject_001_wf/anat_preproc_wf/anat_fit_wf/anat_template_wf/n4_correct/mapflow/_n4_correct*/*.nii.gz`. Let's first load those INU-corrected T1w images.

In [8]:
# Locate the inu-corrected individual T1w images
t1w_files = list((data_path / "inu_corrected_individual_T1w").glob("*_corrected.nii.gz"))
t1w_df = pd.DataFrame({"t1w_file": t1w_files})
t1w_df = t1w_df.assign(session=t1w_df["t1w_file"].astype(str).str.extract(r'ses-(\d+)')[0])

# Locate the reference T1w image
ref_t1 = layout.get(desc="preproc", suffix="T1w", extension=".nii.gz")[0]

assert len(t1w_files) == 36

print(t1w_df.head())

                                            t1w_file session
0  /data/derivatives/hcph-smriprep-reliability/in...     009
1  /data/derivatives/hcph-smriprep-reliability/in...     021
2  /data/derivatives/hcph-smriprep-reliability/in...     025
3  /data/derivatives/hcph-smriprep-reliability/in...     001
4  /data/derivatives/hcph-smriprep-reliability/in...     005


So that the voxel are aligned across sessions, we will use the transformation computed by smriprep to register the T1w images into a common space (the space of global preprocessed T1w image output by smriprep).

In [9]:
# Locate the transformation to register images to the reference T1w
transform_files = layout.get(suffix='xfm', to='T1w', extension='txt', return_type='file')
# 'from' entity in layout.get get misinterpreted
transform_files = [f for f in transform_files if 'from-orig' in f]
transform_df = pd.DataFrame({"transform_file": transform_files})
transform_df = transform_df.assign(session=transform_df["transform_file"].astype(str).str.extract(r'ses-(\d+)')[0])

t1w_df = pd.merge(t1w_df, transform_df, on="session", how="inner")
print(t1w_df["t1w_file"])

0     /data/derivatives/hcph-smriprep-reliability/in...
1     /data/derivatives/hcph-smriprep-reliability/in...
2     /data/derivatives/hcph-smriprep-reliability/in...
3     /data/derivatives/hcph-smriprep-reliability/in...
4     /data/derivatives/hcph-smriprep-reliability/in...
5     /data/derivatives/hcph-smriprep-reliability/in...
6     /data/derivatives/hcph-smriprep-reliability/in...
7     /data/derivatives/hcph-smriprep-reliability/in...
8     /data/derivatives/hcph-smriprep-reliability/in...
9     /data/derivatives/hcph-smriprep-reliability/in...
10    /data/derivatives/hcph-smriprep-reliability/in...
11    /data/derivatives/hcph-smriprep-reliability/in...
12    /data/derivatives/hcph-smriprep-reliability/in...
13    /data/derivatives/hcph-smriprep-reliability/in...
14    /data/derivatives/hcph-smriprep-reliability/in...
15    /data/derivatives/hcph-smriprep-reliability/in...
16    /data/derivatives/hcph-smriprep-reliability/in...
17    /data/derivatives/hcph-smriprep-reliabilit

In [10]:
# Apply the transformation to each T1w image
for index, row in t1w_df.iterrows():  
    # Set up the ApplyTransforms interface
    output_file = str(row['t1w_file']).replace(".nii.gz", "_aligned.nii.gz")
    if not Path(output_file).exists():
        at = ApplyTransforms()
        at.inputs.input_image = str(row['t1w_file'])
        at.inputs.transforms = str(row['transform_file'])
        at.inputs.reference_image = ref_t1
        at.inputs.output_image = str(row['t1w_file']).replace(".nii.gz", "_aligned.nii.gz")
        
        # Run the transformation
        result = at.run()
        print(f"Transformation applied for session {row['session']}, output saved to {output_file}")

Quality Control: Now that we applied the transformation and wrote down the resulting image, verify using brain imaging visualization tool that the images are well aligned.

Once quality control is done, we can extract the WM, GM and CSF voxels from each T1w image using the segmentation probability maps they were computed by smriprep.

In [27]:
# Load segmentation files
csf_seg_file = layout.get(suffix='probseg', space=None, label='CSF', extension='nii.gz', return_type='file')
csf_seg = nib.load(csf_seg_file[0]).get_fdata()
gm_seg_file = layout.get(suffix='probseg', space=None, label='GM', extension='nii.gz', return_type='file')
gm_seg = nib.load(gm_seg_file[0]).get_fdata()
wm_seg_file = layout.get(suffix='probseg', space=None, label='WM', extension='nii.gz', return_type='file')
wm_seg = nib.load(wm_seg_file[0]).get_fdata()

# Binarize the probability maps
thresh = 0.6
wm_seg = wm_seg > thresh
gm_seg = gm_seg > thresh
csf_seg = csf_seg > thresh

assert np.array_equal(np.unique(wm_seg), [0, 1]), "wm_seg is not binary"
assert np.array_equal(np.unique(gm_seg), [0, 1]), "gm_seg is not binary"
assert np.array_equal(np.unique(csf_seg), [0, 1]), "csf_seg is not binary"

# Load the aligned T1w images
t1w_aligned_files = list((data_path / "inu_corrected_individual_T1w").glob("*_aligned.nii.gz"))
assert len(t1w_aligned_files) == 36

# Create a dataframe with session and data extracted from each aligned T1w file
t1w_aligned = []
t1w_wm = []
t1w_gm = []
t1w_csf = []
for file in t1w_aligned_files:
    img = nib.load(file)
    t1w = img.get_fdata()
    t1w_aligned.append(t1w)

    # Apply the masks
    assert t1w.shape == wm_seg.shape
    assert t1w.shape == gm_seg.shape
    assert t1w.shape == csf_seg.shape

    t1w_wm.append(np.multiply(t1w, wm_seg))
    t1w_gm.append(np.multiply(t1w, gm_seg))
    t1w_csf.append(np.multiply(t1w, csf_seg))

assert len(t1w_aligned) == 36
assert t1w_wm[0].shape == wm_seg.shape

In [28]:
t1w_wm = np.array(t1w_wm)
t1w_gm = np.array(t1w_gm)
t1w_csf = np.array(t1w_csf)

print(t1w_wm.shape)

(36, 252, 288, 208)
